In [ ]:
from FIT_python.pipeline_sex.data_import_utils import load_raw_files
from FIT_python.config import RAW_DIR
raw_dfs = load_raw_files(RAW_DIR)
raw_dfs

In [ ]:
# %% [code]
from pathlib import Path
import pandas as pd
import numpy as np
import ast
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import ParameterSampler

from FIT_python.pipeline_sex.pipeline_wrapper_sex import PipelineWrapper
from FIT_python.config import RESULTS_DATA_DIR

# 1) Prepare (Clean → Split → Summary)
prep = PipelineWrapper(
    model_keys=["log_reg"],  # Dummy
    impute_method=None,
    outlier_method=None,
    scaler_method=None,
    reduce_pre_method=None,
    reduce_post_method=None,
    fs_method=None,
    fs_k=None,
)
prep.prepare()



In [2]:
import shutil
from FIT_python.pipeline_sex.pipeline_wrapper_sex import _cache_dir

# Cache-Verzeichnis komplett entfernen
shutil.rmtree(_cache_dir)
print(f"Cache gelöscht: {_cache_dir}")


Cache gelöscht: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\pipeline_cache


In [1]:
# %% [code] RandomizedGridSearchCV mit Speicherung von Roh-Ergebnissen, Summary & Modellen
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, accuracy_score, log_loss
from sklearn.base import clone
from joblib import dump

from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_sex.transform_wrapper import NumericTransformer
from FIT_python.pipeline_sex.feature_selection_wrapper import FeatureSelectionTransformer
from FIT_python.pipeline_sex.outlier_wrapper import OutlierCleanerTransformer
from FIT_python.pipeline_sex.feature_scaler_wrapper import FeatureScalerTransformer
from FIT_python.pipeline_sex.dimensionality_reduction_wrapper import DimensionalityReducerTransformer
from FIT_python.pipeline_sex.models import MODELS
from FIT_python.pipeline_sex.grouped_metrics import individual_accuracies, individual_majority_stats
from FIT_python.config import SPLITS_DIR, RESULTS_DATA_DIR

# 1) Modell‐Keys
model_keys = [
    "logreg_l2","logreg_l1",
    "rf_small","rf_med","rf_large",
    "et_med","et_sm",
    "knn_3","knn_5","knn_7",
    "svm_linear","svm_rbf",
    "xgb_std","xgb_hist",
    "lgbm_std","lgbm_md10",
    "catb_std","catb_fast","lda",
]

# 2) Parameterräume
param_distributions = {
    "select__method":     [None, "forward", "random_forest", "variance", "univariate", "lasso"],
    "select__k":          [1, 2, 3, 5, 10, 20, 50],
    "outlier__method":    [None, "clip", "zscore"],
    "scale__method":      [None, "standard", "robust"],
    "reduce_pre__method": [None, "pca", "umap"],
    "reduce_post__method":[None, "pca", "umap"],
    "clf":                [MODELS[k] for k in model_keys],
}

# 3) Setup Ausgabe‐Ordner & Dateien
out_dir        = Path(RESULTS_DATA_DIR) / "random_search_standard_metrics"
raw_csv        = out_dir / "raw_results.csv"
summary_csv    = out_dir / "best_results.csv"
best_bal_dir   = out_dir / "best_balanced"
best_ind_dir   = out_dir / "best_individual"
for d in (out_dir, best_bal_dir, best_ind_dir):
    d.mkdir(parents=True, exist_ok=True)

# 4) Loop über Spezies
first_raw = True
summary_records = []

for species_dir in tqdm(sorted(Path(SPLITS_DIR).iterdir()), desc="Species"):
    if not species_dir.is_dir(): 
        continue
    species = species_dir.name

    # 4.1) Daten laden
    df_train = pd.read_parquet(species_dir / "train.parquet").query("sex in ['f','m']")
    df_test  = pd.read_parquet(species_dir / "test.parquet").query("sex in ['f','m']")
    for df in (df_train, df_test):
        if "Fold" in df.columns:
            df.drop(columns=["Fold"], inplace=True)

    X_tr, y_tr = df_train.select_dtypes(include="number"), df_train["sex"].map({"f":0,"m":1}).values
    ids_tr     = df_train["individual_id"].values
    X_te, y_te = df_test.select_dtypes(include="number"),  df_test["sex"].map({"f":0,"m":1}).values
    ids_te     = df_test["individual_id"].values

    # 4.2) Basis‐Pipeline
    steps = [
        ("transform",  NumericTransformer()),
        ("select",     FeatureSelectionTransformer(method="forward", k=1)),
        ("outlier",    OutlierCleanerTransformer(method="clip")),
        ("scale",      FeatureScalerTransformer(method="standard")),
        ("reduce_pre", DimensionalityReducerTransformer(method="pca", n_components=1)),
        ("reduce_post",DimensionalityReducerTransformer(method="pca", n_components=1)),
        ("clf",        MODELS["rf_small"]),
    ]
    pipe = Pipeline(steps)

    # 4.3) RandomizedSearchCV mit Standard-Metriken
    scoring = {
        "accuracy":          "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "neg_log_loss":      "neg_log_loss"
    }
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=10,
        scoring=scoring,
        refit=False,
        cv=5,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    # 4.4) Fit auf Train mit tqdm
    total_fits = 10 * 5
    with tqdm_joblib(tqdm(desc=f"{species} RS-CV", total=total_fits, leave=False)):
        search.fit(X_tr, y_tr)

    # 4.5) Alle Kandidaten evaluieren
    records = []
    for params in search.cv_results_["params"]:
        # eindeutige Pipeline-ID
        pipeline_id = "|".join(f"{k}={params[k]}" for k in sorted(params))
        # Modell trainieren & vorhersagen
        model = clone(pipe).set_params(**params).fit(X_tr, y_tr)
        y_tr_pred = model.predict(X_tr)
        y_te_pred = model.predict(X_te)
        y_all_true = np.concatenate([y_tr, y_te])
        y_all_pred = np.concatenate([y_tr_pred, y_te_pred])
        ids_all     = np.concatenate([ids_tr, ids_te])

        # Custom‐Metriken
        fem_tr, mal_tr, bal_tr = individual_accuracies(y_tr, y_tr_pred, ids_tr)
        fem_te, mal_te, bal_te = individual_accuracies(y_te, y_te_pred, ids_te)
        fem_all, mal_all, bal_all = individual_accuracies(y_all_true, y_all_pred, ids_all)
        ct_tr, wr_tr, pct_tr = individual_majority_stats(y_tr, y_tr_pred, ids_tr)
        ct_te, wr_te, pct_te = individual_majority_stats(y_te, y_te_pred, ids_te)
        ct_all,wr_all,pct_all= individual_majority_stats(y_all_true, y_all_pred, ids_all)

        rec = {
            "species":            species,
            "pipeline_id":        pipeline_id,
            **params,
            "female_train_acc":   fem_tr,
            "male_train_acc":     mal_tr,
            "balanced_train_acc": bal_tr,
            "female_test_acc":    fem_te,
            "male_test_acc":      mal_te,
            "balanced_test_acc":  bal_te,
            "female_full_acc":    fem_all,
            "male_full_acc":      mal_all,
            "balanced_full_acc":  bal_all,
            "maj_train_count":    ct_tr,
            "maj_train_wrong":    wr_tr,
            "maj_train_pct":      pct_tr,
            "maj_test_count":     ct_te,
            "maj_test_wrong":     wr_te,
            "maj_test_pct":       pct_te,
            "maj_full_count":     ct_all,
            "maj_full_wrong":     wr_all,
            "maj_full_pct":       pct_all,
        }
        records.append(rec)

    df_eval = pd.DataFrame(records)
    # Roh-Ergebnisse anhängen
    df_eval.to_csv(raw_csv, mode="a", header=first_raw, index=False)
    first_raw = False

    # 4.6) Bestes Modell bestimmen
    best_maj_idx  = df_eval["maj_test_pct"].idxmax()
    best_bal_idx  = df_eval["balanced_full_acc"].idxmax()
    best_majority= df_eval.loc[best_maj_idx]
    best_balanced= df_eval.loc[best_bal_idx]

    # 4.7) Modelle abspeichern
    model_maj = clone(pipe).set_params(**{k: best_majority[k] for k in param_distributions}).fit(X_tr, y_tr)
    dump(model_maj, best_ind_dir / f"{species}.joblib")
    model_bal = clone(pipe).set_params(**{k: best_balanced[k] for k in param_distributions}).fit(X_tr, y_tr)
    dump(model_bal, best_bal_dir / f"{species}.joblib")

    # Summary‐Einträge vorbereiten
    rec_maj = best_majority.to_dict(); rec_maj["best_for"] = "individual"; summary_records.append(rec_maj)
    rec_bal = best_balanced.to_dict(); rec_bal["best_for"] = "balanced";  summary_records.append(rec_bal)

# 5) Summary speichern
pd.DataFrame(summary_records).to_csv(summary_csv, index=False)

print("✅ Fertig: Ergebnisse, Summaries und Best-Modelle gespeichert.")


Species:   0%|          | 0/10 [00:00<?, ?it/s]

aj_someringii RS-CV:   0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


amur_tiger RS-CV:   0%|          | 0/50 [00:00<?, ?it/s]

...

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


white_rhino RS-CV:   0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ Fertig: Ergebnisse, Summaries und Best-Modelle gespeichert.


In [ ]:
print("Train cols:", X_tr.columns.tolist())
print("Test cols: ", X_te.columns.tolist())


In [ ]:
import pandas as pd
from joblib import load

# Pfade definieren
model_path = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/sex_models_best/eurasian_otter.joblib"
data_path  = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/raw/Eurasian Otter.csv"
output_csv = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/eurasian_otter_predictions.csv"

# Modell laden
clf = load(model_path)

# Daten laden und Spaltennamen anpassen (Punkte, Bindestriche → Unterstriche; T → t)
df = pd.read_csv(data_path)
df.columns = [
    col.replace('.', '_')
       .replace('-', '_')
       .replace('T', 't')
    for col in df.columns
]

# Direkt Vorhersagen aus der Pipeline holen
df["predicted_sex"]     = clf.predict(df)
probas = clf.predict_proba(df)
df["predicted_proba_f"] = probas[:, 0]
df["predicted_proba_m"] = probas[:, 1]

# Ergebnisse als CSV speichern
df.to_csv(output_csv, index=False)
print(f"Ergebnisse gespeichert in: {output_csv}")
